In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 32 samples and 270 columns.
First few rows of the data:


,living_day,die,Etiology,Degree_of_tumor_differentiation,AFP(ng/mL),TNM_stages,BCLC,Tumor_size(mm),age,Satellites,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,176,1,Cirrhosis,Poor,30783,Ⅳ,B,105,37,Present,...,4.536021,2.249830,3.781962,5.140040,3.470682,3.293235,4.711250,4.434599,2.847444,4.243018
1,190,1,Hepatitis_B,Poor,2668,Ⅳ,C,190,44,Present,...,3.924160,1.357657,2.073493,5.633265,2.004485,2.394021,3.718035,3.279829,1.996515,3.182071
2,4,1,Hepatitis_B,Poor,>121000,Ⅲ,C,230,43,Absent,...,6.147698,3.642578,6.042068,6.485741,5.608361,5.945016,7.058042,6.250611,4.534191,5.931232
3,519,1,Hepatitis_B,Poor,1.01,Ⅳ,C,130,64,Absent,...,3.920815,1.615761,3.311212,4.523287,2.619709,2.821889,4.293913,3.627103,2.215886,3.478311
4,533,1,Hepatitis_B,Poor,3710,Ⅳ,C,120,48,Absent,...,4.526706,2.097380,4.076484,5.227762,3.436344,3.337557,5.054281,4.396321,2.804797,4.459888


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for living_day or die: 0
Number of samples with complete data for both variables: 32
Number of censored samples (event_var == 0.0): 7
Number of events (event_var == 1.0): 25
Censored ratio (among complete): 7 / 32 = 0.219 (21.9%)
Skipping stats for PFS: columns not in preprocessed table ('PFS_day', 'unknown').
Updated stats for 1 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE227378/description.json
